In [ ]:
# a lot of ignore warnings were creating noisy outputs, so filter these away.
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

In [ ]:
# Load BESSTIE dataset and view overall stats
from datasets import load_dataset
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

dataset = load_dataset("surrey-nlp/BESSTIE-CW-26")

print(dataset)

training_dataframe = dataset['train'].to_pandas()
print(training_dataframe['Sentiment'].value_counts())
print(training_dataframe['Sarcasm'].value_counts())
print(training_dataframe['variety'].value_counts())




In [ ]:
# What is the optimal number to use for max_features?
from sklearn.metrics import f1_score
import numpy as np

test_dataframe = dataset['test'].to_pandas()
seeds = [1, 2, 3]

for max_features in [1000, 1500, 2000, 5000, 10000, 20000]:
    vectorizer = TfidfVectorizer(max_features=max_features)
    X_train = vectorizer.fit_transform(training_dataframe['text'])
    X_test = vectorizer.transform(test_dataframe['text'])

    sentiment_scores = []
    sarcasm_scores = []
    for seed in seeds:
        lr_sentiment = LogisticRegression(max_iter=1000, random_state=seed)
        lr_sentiment.fit(X_train, training_dataframe['Sentiment'])
        sentiment_scores.append(f1_score(test_dataframe['Sentiment'], lr_sentiment.predict(X_test), average='macro'))

        lr_sarcasm = LogisticRegression(max_iter=1000, random_state=seed)
        lr_sarcasm.fit(X_train, training_dataframe['Sarcasm'])
        sarcasm_scores.append(f1_score(test_dataframe['Sarcasm'], lr_sarcasm.predict(X_test), average='macro'))

    print(f"max_features={max_features:>6} | Sentiment F1: {np.mean(sentiment_scores):.4f} ± {np.std(sentiment_scores):.4f} | Sarcasm F1: {np.mean(sarcasm_scores):.4f} ± {np.std(sarcasm_scores):.4f}")

Accuracy is the wrong metric for our purposes becuase for sarcasm the dataset is highly imbalanced. The 0.86 accuracy is misleading. The precision and recall for sarcasm is 0.00. The macro score is more useful.

1,500 is the smallest vocabulary found with the highest macro-F1. The below explores the effects of applying combinations of bigrams/unigrams and with and without sublinear TF


In [ ]:
# sentiment with 10,000 features, unigrams, and sublinear TF

test_dataframe = dataset['test'].to_pandas()

vectorizer = TfidfVectorizer(max_features=10000, sublinear_tf=True)
X_train = vectorizer.fit_transform(training_dataframe['text'])
X_test = vectorizer.transform(test_dataframe['text'])

# Sentiment
lr_sentiment = LogisticRegression(max_iter=1000)
lr_sentiment.fit(X_train, training_dataframe['Sentiment'])
print("Sentiment:")
print(classification_report(test_dataframe['Sentiment'], lr_sentiment.predict(X_test)))

The following experiments with various combinations of max features, ngrams, default and sublinear TF

In [ ]:
from sklearn.metrics import f1_score
import numpy as np

configs = [
    {"name": "unigrams, default TF",   "ngram_range": (1,1), "sublinear_tf": False},
    {"name": "unigrams, sublinear TF", "ngram_range": (1,1), "sublinear_tf": True},
    {"name": "bigrams, default TF",    "ngram_range": (1,2), "sublinear_tf": False},
    {"name": "bigrams, sublinear TF",  "ngram_range": (1,2), "sublinear_tf": True},
]

seeds = [1, 2, 3]
results = []

for max_features in [1000, 1500, 2000, 5000, 10000, 20000]:
    for config in configs:
        vectorizer = TfidfVectorizer(
            max_features=max_features,
            ngram_range=config["ngram_range"],
            sublinear_tf=config["sublinear_tf"],
        )
        X_train = vectorizer.fit_transform(training_dataframe['text'])
        X_test = vectorizer.transform(test_dataframe['text'])

        sentiment_scores = []
        sarcasm_scores = []
        for seed in seeds:
            lr_sentiment = LogisticRegression(max_iter=1000, random_state=seed).fit(X_train, training_dataframe['Sentiment'])
            lr_sarcasm = LogisticRegression(max_iter=1000, random_state=seed).fit(X_train, training_dataframe['Sarcasm'])
            sentiment_scores.append(f1_score(test_dataframe['Sentiment'], lr_sentiment.predict(X_test), average='macro'))
            sarcasm_scores.append(f1_score(test_dataframe['Sarcasm'], lr_sarcasm.predict(X_test), average='macro'))

        results.append({
            "max_features": max_features,
            "config": config["name"],
            "sentiment mean": np.mean(sentiment_scores),
            "sentiment std": np.std(sentiment_scores),
            "sarcasm mean": np.mean(sarcasm_scores),
            "sarcasm std": np.std(sarcasm_scores),
        })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

In [ ]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
print(sorted(ENGLISH_STOP_WORDS))
print(f"\nTotal: {len(ENGLISH_STOP_WORDS)} words")

The best result is unigrams with sublinear TF at max_features=20,000 (Sentiment mean of 0.83727).

At every vocabulary size, bigrams are slightly worse than unigrams.
This finding is consistent with the commentary by Bekkerman and Allan, 2003 regarding Reuters and 20 Newsgroups datasets. They found at best the use of bigrams produces an improvement which is "statistically insignificant".
Using Bigrams in Text Categorization by Bekkerman and Allan, 2003
https://ciir.cs.umass.edu/pubfiles/ir-408.pdf

The above enumeration of stopwords reveals many words such as 'no', 'nor', 'not' that are essential for understanding sentiment and sarcasm. Therefore we reject stopword removal.  

Next we apply Logistic Regression with balanced class weight.

In [ ]:
# Balanced class weight
lr_sarcasm = LogisticRegression(max_iter=1000, class_weight='balanced')
lr_sarcasm.fit(X_train, training_dataframe['Sarcasm'])
sarcasm_predictions = lr_sarcasm.predict(X_test)
print("Test dataset Logistic Regression Sarcasm metrics with balanced class weight")
print(classification_report(test_dataframe['Sarcasm'], sarcasm_predictions))


The macro F1 score substantially increases from 0.46 to 0.62

The accuracy and weighted averages are slightly lower, but the macro avg is the most important metric.

Next we compare the balanced class with our own class weight ratio sweep.

For details on class weights see "Logistic Regression in Rare Events Data" by King & Zeng, 2001

In [ ]:
from sklearn.metrics import precision_score, recall_score

vectorizer = TfidfVectorizer(max_features=10000, sublinear_tf=True)
X_train = vectorizer.fit_transform(training_dataframe['text'])
X_test = vectorizer.transform(test_dataframe['text'])

# examples from class 0 (sentiment) count with weight 1 in all cases. For examples from class 1 (sarcasm) experiment with values 2, 3, 4, 5, 6 and 8.
results = []
for weight in ['balanced', {0: 1, 1: 2}, {0: 1, 1: 3}, {0: 1, 1: 4}, {0: 1, 1: 5}, {0: 1, 1: 6}, {0: 1, 1: 8}]:
    lr = LogisticRegression(max_iter=1000, class_weight=weight)
    lr.fit(X_train, training_dataframe['Sarcasm'])
    preds = lr.predict(X_test)
    results.append({
        'weight': str(weight),
        'macro_f1': f1_score(test_dataframe['Sarcasm'], preds, average='macro'),
        'sarc_f1': f1_score(test_dataframe['Sarcasm'], preds, pos_label=1),
        'sarc_precision': precision_score(test_dataframe['Sarcasm'], preds, pos_label=1),
        'sarc_recall': recall_score(test_dataframe['Sarcasm'], preds, pos_label=1),
    })

import pandas as pd
pd.DataFrame(results)

The balanced class has the highest recall and f1 scores for sarcasm, confirming it is essentially optimal. The results demonstrate the precision/recall tradeoff.

GloVe analysis

globe.6B.zip is a large file (822MB) and it can take a few minutes to unzip globe.6B.300d.txt

In [ ]:
import os, urllib.request, zipfile

def reporthook(block, block_size, total):
    downloaded = block * block_size
    if total > 0:
        pct = min(downloaded / total * 100, 100)
        print(f'\r  {pct:.1f}%  {downloaded/1e6:.1f} / {total/1e6:.1f} MB', end='', flush=True)

glove_zip = 'glove.6B.zip'

if not os.path.exists('glove.6B.300d.txt'):
    if os.path.exists(glove_zip) and not zipfile.is_zipfile(glove_zip):
        print("Removing corrupt zip file...")
        os.remove(glove_zip)

    if not os.path.exists(glove_zip):
        print("Downloading GloVe embeddings (822 MB)...")
        urllib.request.urlretrieve('http://nlp.stanford.edu/data/glove.6B.zip', glove_zip, reporthook)
        print()

    print("Extracting...")
    with zipfile.ZipFile(glove_zip, 'r') as z:
        z.extractall('.')
    print("Done.")
else:
    print("GloVe already extracted, skipping download.")

In [ ]:
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score

# Load GloVe 300d
glove = {}
with open('glove.6B.300d.txt', 'r') as f:
    for line in f:
        parts = line.split()
        glove[parts[0]] = np.array(parts[1:], dtype=np.float32)

def doc_to_glove(text, glove, dim=300):
    words = text.lower().split()
    vecs = [glove[w] for w in words if w in glove]
    if not vecs:
        return np.zeros(dim)
    return np.mean(vecs, axis=0)

X_train_glove = np.array([doc_to_glove(t, glove) for t in training_dataframe['text']])
X_test_glove = np.array([doc_to_glove(t, glove) for t in test_dataframe['text']])

seeds = [1, 2]
sentiment_scores, sarcasm_unbal_scores, sarcasm_bal_scores = [], [], []

for seed in seeds:
    lr = LogisticRegression(max_iter=1000, random_state=seed)
    lr.fit(X_train_glove, training_dataframe['Sentiment'])
    sentiment_scores.append(f1_score(test_dataframe['Sentiment'], lr.predict(X_test_glove), average='macro'))

    lr = LogisticRegression(max_iter=1000, random_state=seed)
    lr.fit(X_train_glove, training_dataframe['Sarcasm'])
    sarcasm_unbal_scores.append(f1_score(test_dataframe['Sarcasm'], lr.predict(X_test_glove), average='macro'))

    lr = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=seed)
    lr.fit(X_train_glove, training_dataframe['Sarcasm'])
    sarcasm_bal_scores.append(f1_score(test_dataframe['Sarcasm'], lr.predict(X_test_glove), average='macro'))

print(f"GloVe 300d + LR  Sentiment:       {np.mean(sentiment_scores):.4f} ± {np.std(sentiment_scores):.4f}")
print(f"GloVe 300d + LR  Sarcasm (unbal): {np.mean(sarcasm_unbal_scores):.4f} ± {np.std(sarcasm_unbal_scores):.4f}")
print(f"GloVe 300d + LR  Sarcasm (bal):   {np.mean(sarcasm_bal_scores):.4f} ± {np.std(sarcasm_bal_scores):.4f}")

GloVe comes in 50, 100, 200, and 300 dimensional versions. For completeness let's sweep all sizes.

In [ ]:
import pandas as pd
from sklearn.metrics import f1_score

seeds = [1, 2]
results = []

for dim in [50, 100, 200, 300]:
    glove = {}
    with open(f'glove.6B.{dim}d.txt', 'r') as f:
        for line in f:
            parts = line.split()
            glove[parts[0]] = np.array(parts[1:], dtype=np.float32)

    X_train_g = np.array([doc_to_glove(t, glove, dim=dim) for t in training_dataframe['text']])
    X_test_g = np.array([doc_to_glove(t, glove, dim=dim) for t in test_dataframe['text']])

    sentiment_scores, sarcasm_unbal_scores, sarcasm_bal_scores = [], [], []
    for seed in seeds:
        lr = LogisticRegression(max_iter=1000, random_state=seed)
        lr.fit(X_train_g, training_dataframe['Sentiment'])
        sentiment_scores.append(f1_score(test_dataframe['Sentiment'], lr.predict(X_test_g), average='macro'))

        lr = LogisticRegression(max_iter=1000, random_state=seed)
        lr.fit(X_train_g, training_dataframe['Sarcasm'])
        sarcasm_unbal_scores.append(f1_score(test_dataframe['Sarcasm'], lr.predict(X_test_g), average='macro'))

        lr = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=seed)
        lr.fit(X_train_g, training_dataframe['Sarcasm'])
        sarcasm_bal_scores.append(f1_score(test_dataframe['Sarcasm'], lr.predict(X_test_g), average='macro'))

    results.append({
        'dim': dim,
        'sentiment mean': np.mean(sentiment_scores),
        'sentiment std': np.std(sentiment_scores),
        'sarcasm unbal mean': np.mean(sarcasm_unbal_scores),
        'sarcasm unbal std': np.std(sarcasm_unbal_scores),
        'sarcasm bal mean': np.mean(sarcasm_bal_scores),
        'sarcasm bal std': np.std(sarcasm_bal_scores),
    })

pd.DataFrame(results)

GloVe + LR Sentiment and GloVe + LR Sarcasm (balanced and unbalanced) results are worse than the equivalent LR tests with TF-IDF.

Next we evaluate SVM:

In [ ]:
from sklearn.svm import LinearSVC
from sklearn.metrics import f1_score
import numpy as np

seeds = [1, 2]
sentiment_scores, sarcasm_scores = [], []

for seed in seeds:
    svm = LinearSVC(max_iter=10000, random_state=seed)
    svm.fit(X_train, training_dataframe['Sentiment'])
    sentiment_scores.append(f1_score(test_dataframe['Sentiment'], svm.predict(X_test), average='macro'))

    svm = LinearSVC(max_iter=10000, class_weight='balanced', random_state=seed)
    svm.fit(X_train, training_dataframe['Sarcasm'])
    sarcasm_scores.append(f1_score(test_dataframe['Sarcasm'], svm.predict(X_test), average='macro'))

print(f"SVM  Sentiment:       {np.mean(sentiment_scores):.4f} ± {np.std(sentiment_scores):.4f}")
print(f"SVM  Sarcasm (bal):   {np.mean(sarcasm_scores):.4f} ± {np.std(sarcasm_scores):.4f}")

With sublinear TF

In [ ]:
from sklearn.svm import LinearSVC
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import f1_score
import numpy as np

vectorizer = TfidfVectorizer(max_features=10000, sublinear_tf=True)
X_train = vectorizer.fit_transform(training_dataframe['text'])
X_test = vectorizer.transform(test_dataframe['text'])

seeds = [1, 2]
sentiment_scores, sarcasm_unbal_scores, sarcasm_bal_scores = [], [], []

for seed in seeds:
    svm = LinearSVC(max_iter=10000, random_state=seed)
    svm.fit(X_train, training_dataframe['Sentiment'])
    sentiment_scores.append(f1_score(test_dataframe['Sentiment'], svm.predict(X_test), average='macro'))

    svm = LinearSVC(max_iter=10000, random_state=seed)
    svm.fit(X_train, training_dataframe['Sarcasm'])
    sarcasm_unbal_scores.append(f1_score(test_dataframe['Sarcasm'], svm.predict(X_test), average='macro'))

    svm = LinearSVC(max_iter=10000, class_weight='balanced', random_state=seed)
    svm.fit(X_train, training_dataframe['Sarcasm'])
    sarcasm_bal_scores.append(f1_score(test_dataframe['Sarcasm'], svm.predict(X_test), average='macro'))

print(f"SVM (sublinear TF)  Sentiment:       {np.mean(sentiment_scores):.4f} ± {np.std(sentiment_scores):.4f}")
print(f"SVM (sublinear TF)  Sarcasm (unbal): {np.mean(sarcasm_unbal_scores):.4f} ± {np.std(sarcasm_unbal_scores):.4f}")
print(f"SVM (sublinear TF)  Sarcasm (bal):   {np.mean(sarcasm_bal_scores):.4f} ± {np.std(sarcasm_bal_scores):.4f}")

In [ ]:
!pip install --upgrade transformers

RoBERTa sentiment analysis

In [ ]:
# this code is sentiment specific and can be removed later because what follows is a refactored version of this code for sentiment and sarcasm
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
import numpy as np

model_name = "roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(batch['text'], padding='max_length', truncation=True, max_length=128)

tokenized_train = dataset['train'].map(tokenize, batched=True)
tokenized_test = dataset['test'].map(tokenize, batched=True)
tokenized_val = dataset['validation'].map(tokenize, batched=True)

tokenized_train = tokenized_train.map(lambda examples: {'labels': [int(s) for s in examples['Sentiment']]}, batched=True, remove_columns=['Sentiment'])
tokenized_test = tokenized_test.map(lambda examples: {'labels': [int(s) for s in examples['Sentiment']]}, batched=True, remove_columns=['Sentiment'])
tokenized_val = tokenized_val.map(lambda examples: {'labels': [int(s) for s in examples['Sentiment']]}, batched=True, remove_columns=['Sentiment'])

tokenized_train.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
tokenized_test.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
tokenized_val.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2, problem_type="single_label_classification")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    from sklearn.metrics import classification_report
    print(classification_report(labels, preds))
    from sklearn.metrics import f1_score
    return {'macro_f1': f1_score(labels, preds, average='macro')}

# Define TrainingArguments
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    save_total_limit=1,
)

# Instantiate Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    compute_metrics=compute_metrics,
)

trainer.train()

Test set evaluation

In [ ]:
from sklearn.metrics import f1_score

sentiment_scores = []
for seed in [1, 2]:
    print(f"\n{'='*40}\nSeed: {seed}\n{'='*40}")

    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

    training_args = TrainingArguments(
        output_dir=f'./results/seed_{seed}',
        num_train_epochs=3,
        seed=seed,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='macro_f1',
        save_total_limit=1,
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_val,
        compute_metrics=compute_metrics,
    )

    trainer.train()
    predictions = trainer.predict(tokenized_test)
    preds = np.argmax(predictions.predictions, axis=-1)
    score = f1_score(tokenized_test['labels'], preds, average='macro')
    sentiment_scores.append(score)
    print(f"Seed {seed} Macro-F1: {score:.4f}")

print(f"\nRoBERTa Sentiment: {np.mean(sentiment_scores):.4f} ± {np.std(sentiment_scores):.4f}")

Common methods for RoBERTa analysis

In [ ]:
from sklearn.metrics import f1_score
from sklearn.metrics import classification_report

# Common methods
def tokenize(batch):
    return tokenizer(batch['text'], padding='max_length', truncation=True, max_length=128)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    print(classification_report(labels, preds))
    return {'macro_f1': f1_score(labels, preds, average='macro')}

def prepare_dataset(split, task):
    tokenized = split.map(tokenize, batched=True)
    tokenized = tokenized.map(
        lambda examples: {'labels': [int(s) for s in examples[task]]},
        batched=True, remove_columns=[task]
    )
    tokenized.set_format('torch', columns=['input_ids', 'attention_mask', 'labels'])
    return tokenized

# 3 epochs is a common default for fine-tuning. We could try a higher number of epochs as a further investigation.
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    save_total_limit=1,
)

RoBERTA Sarcasm analysis

In [ ]:
from sklearn.metrics import f1_score
import numpy as np

train_sentiment = prepare_dataset(dataset['train'], 'Sentiment')
test_sentiment = prepare_dataset(dataset['test'], 'Sentiment')
val_sentiment = prepare_dataset(dataset['validation'], 'Sentiment')

sentiment_scores = []
for seed in [1, 2]:
    print(f"\n{'='*40}\nSeed: {seed}\n{'='*40}")

    sentiment_model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=2, problem_type="single_label_classification"
    )

    training_args = TrainingArguments(
        output_dir=f'./results/seed_{seed}',
        num_train_epochs=3,
        seed=seed,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='macro_f1',
        save_total_limit=1,
    )

    trainer = Trainer(
        model=sentiment_model,
        args=training_args,
        train_dataset=train_sentiment,
        eval_dataset=val_sentiment,
        compute_metrics=compute_metrics,
    )
    trainer.train()
    predictions = trainer.predict(test_sentiment)
    preds = np.argmax(predictions.predictions, axis=-1)
    score = f1_score(test_sentiment['labels'], preds, average='macro')
    sentiment_scores.append(score)
    print(f"Seed {seed} Macro-F1: {score:.4f}")

print(f"\nSentiment: {np.mean(sentiment_scores):.4f} ± {np.std(sentiment_scores):.4f}")

In [ ]:
from sklearn.metrics import f1_score
import numpy as np

train_sentiment = prepare_dataset(dataset['train'], 'Sentiment')
test_sentiment = prepare_dataset(dataset['test'], 'Sentiment')
val_sentiment = prepare_dataset(dataset['validation'], 'Sentiment')

sentiment_scores = []
for seed in [1, 2]:
    print(f"\n{'='*40}\nSeed: {seed}\n{'='*40}")

    sentiment_model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=2, problem_type="single_label_classification"
    )

    training_args = TrainingArguments(
        output_dir=f'./results/seed_{seed}',
        num_train_epochs=3,
        seed=seed,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='macro_f1',
        save_total_limit=1,
    )

    trainer = Trainer(
        model=sentiment_model,
        args=training_args,
        train_dataset=train_sentiment,
        eval_dataset=val_sentiment,
        compute_metrics=compute_metrics,
    )
    trainer.train()
    predictions = trainer.predict(test_sentiment)
    preds = np.argmax(predictions.predictions, axis=-1)
    score = f1_score(test_sentiment['labels'], preds, average='macro')
    sentiment_scores.append(score)
    print(f"Seed {seed} Macro-F1: {score:.4f}")

print(f"\nSentiment: {np.mean(sentiment_scores):.4f} ± {np.std(sentiment_scores):.4f}")

The results show a problem with the sarcasm model. Almost everything is being predicted as not sarcasm.

So the model needs adjusting to account for the class imbalance.

In [ ]:
from torch import nn
import torch
from sklearn.metrics import f1_score
import numpy as np

train_sarcasm = prepare_dataset(dataset['train'], 'Sarcasm')
test_sarcasm = prepare_dataset(dataset['test'], 'Sarcasm')
val_sarcasm = prepare_dataset(dataset['validation'], 'Sarcasm')

class WeightedTrainer(Trainer):
    # override the Trainer compute_loss
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        # model expects only the text inputs, so remove labels
        labels = inputs.pop("labels")
        # pass the text inputs through the model and get the logits outputs
        outputs = model(**inputs)
        logits = outputs.logits

        # divide the number of non-sarcasm examples by sarcasm examples for class 1 (sarcasm). This is about 6.15
        sarcasm_ratio = len(training_dataframe[training_dataframe['Sarcasm']==0]) / len(training_dataframe[training_dataframe['Sarcasm']==1])
        # create a weight tensor with two values, 1.0 for class 0 (not sarcasm) and the sarcasm_ratio for class 1
        weight = torch.tensor([1.0, sarcasm_ratio]).to(logits.device)
        loss_fn = nn.CrossEntropyLoss(weight=weight)
        # calculate the loss by comparing the model predictions against the actual labels
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

sarcasm_scores = []
for seed in [1, 2]:
    print(f"\n{'='*40}\nSeed: {seed}\n{'='*40}")

    sarcasm_model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=2, problem_type="single_label_classification"
    )

    training_args_seed = TrainingArguments(
        output_dir=f'./results/seed_{seed}',
        num_train_epochs=3,
        seed=seed,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='macro_f1',
        save_total_limit=1,
    )

    trainer_sarcasm = WeightedTrainer(
        model=sarcasm_model,
        args=training_args_seed,
        train_dataset=train_sarcasm,
        eval_dataset=val_sarcasm,
        compute_metrics=compute_metrics,
    )

    trainer_sarcasm.train()
    predictions_sarcasm = trainer_sarcasm.predict(test_sarcasm)
    preds = np.argmax(predictions_sarcasm.predictions, axis=-1)
    score = f1_score(test_sarcasm['labels'], preds, average='macro')
    sarcasm_scores.append(score)
    print(f"Seed {seed} Macro-F1: {score:.4f}")

print(f"\nWeightedTrainer Sarcasm: {np.mean(sarcasm_scores):.4f} ± {np.std(sarcasm_scores):.4f}")


In [ ]:
from torch import nn
import torch
from sklearn.metrics import f1_score
import numpy as np

train_sarcasm = prepare_dataset(dataset['train'], 'Sarcasm')
test_sarcasm = prepare_dataset(dataset['test'], 'Sarcasm')
val_sarcasm = prepare_dataset(dataset['validation'], 'Sarcasm')

class WeightedTrainer(Trainer):
    # override the Trainer compute_loss
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        # divide the number of non-sarcasm examples by sarcasm examples for class 1 (sarcasm). This is about 6.15
        sarcasm_ratio = len(training_dataframe[training_dataframe['Sarcasm']==0]) / len(training_dataframe[training_dataframe['Sarcasm']==1])
        weight = torch.tensor([1.0, sarcasm_ratio]).to(logits.device)
        loss_fn = nn.CrossEntropyLoss(weight=weight)
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

sarcasm_scores = []
for seed in [1, 2]:
    print(f"\n{'='*40}\nSeed: {seed}\n{'='*40}")

    sarcasm_model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=2, problem_type="single_label_classification"
    )

    training_args_seed = TrainingArguments(
        output_dir=f'./results/seed_{seed}',
        num_train_epochs=3,
        seed=seed,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='macro_f1',
        save_total_limit=1,
    )

    trainer_sarcasm = WeightedTrainer(
        model=sarcasm_model,
        args=training_args_seed,
        train_dataset=train_sarcasm,
        eval_dataset=val_sarcasm,
        compute_metrics=compute_metrics,
    )

    trainer_sarcasm.train()
    predictions_sarcasm = trainer_sarcasm.predict(test_sarcasm)
    preds = np.argmax(predictions_sarcasm.predictions, axis=-1)
    score = f1_score(test_sarcasm['labels'], preds, average='macro')
    sarcasm_scores.append(score)
    print(f"Seed {seed} Macro-F1: {score:.4f}")

print(f"\nWeightedTrainer Sarcasm: {np.mean(sarcasm_scores):.4f} ± {np.std(sarcasm_scores):.4f}")


In [ ]:
from torch import nn
import torch
from sklearn.metrics import f1_score
import numpy as np

class WeightedTrainer(Trainer):
    # override the Trainer compute_loss
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        sarcasm_ratio = 3
        weight = torch.tensor([1.0, sarcasm_ratio]).to(logits.device)
        loss_fn = nn.CrossEntropyLoss(weight=weight)
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

sarcasm_scores = []
for seed in [1, 2]:
    print(f"\n{'='*40}\nSeed: {seed}\n{'='*40}")

    sarcasm_model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=2, problem_type="single_label_classification"
    )

    # reduce the learning rate from the default 5e-5 to 1e-5
    training_args_sarcasm = TrainingArguments(
        output_dir=f'./results_sarcasm/seed_{seed}',
        num_train_epochs=5,
        seed=seed,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        learning_rate=1e-5,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='macro_f1',
        save_total_limit=1,
    )

    trainer_sarcasm = WeightedTrainer(
        model=sarcasm_model,
        args=training_args_sarcasm,
        train_dataset=train_sarcasm,
        eval_dataset=val_sarcasm,
        compute_metrics=compute_metrics,
    )

    trainer_sarcasm.train()
    predictions_sarcasm = trainer_sarcasm.predict(test_sarcasm)
    preds = np.argmax(predictions_sarcasm.predictions, axis=-1)
    score = f1_score(test_sarcasm['labels'], preds, average='macro')
    sarcasm_scores.append(score)
    print(f"Seed {seed} Macro-F1: {score:.4f}")

print(f"\nWeightedTrainer (ratio=3, lr=1e-5) Sarcasm: {np.mean(sarcasm_scores):.4f} ± {np.std(sarcasm_scores):.4f}")


RoBERTa performs much better for sentiment, but only slightly better for sarcasm.

To confirm whether the lower LR is needed, run the 3x weight with the default LR:

In [ ]:
from sklearn.metrics import f1_score
import numpy as np

sarcasm_scores = []
for seed in [1, 2]:
    print(f"\n{'='*40}\nSeed: {seed}\n{'='*40}")

    sarcasm_model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=2, problem_type="single_label_classification"
    )

    training_args_seed = TrainingArguments(
        output_dir=f'./results/seed_{seed}',
        num_train_epochs=5,
        seed=seed,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='macro_f1',
        save_total_limit=1,
    )

    trainer_sarcasm = WeightedTrainer(
        model=sarcasm_model,
        args=training_args_seed,
        train_dataset=train_sarcasm,
        eval_dataset=val_sarcasm,
        compute_metrics=compute_metrics,
    )

    trainer_sarcasm.train()
    predictions_sarcasm = trainer_sarcasm.predict(test_sarcasm)
    preds = np.argmax(predictions_sarcasm.predictions, axis=-1)
    score = f1_score(test_sarcasm['labels'], preds, average='macro')
    sarcasm_scores.append(score)
    print(f"Seed {seed} Macro-F1: {score:.4f}")

print(f"\nWeightedTrainer (ratio=3, default lr) Sarcasm: {np.mean(sarcasm_scores):.4f} ± {np.std(sarcasm_scores):.4f}")


These results show degraded performance in the form of unstable training. The test set result of 0.62 is the same. But using a lower LR is preferred.

Let's try 2e-5

In [ ]:
from sklearn.metrics import f1_score
import numpy as np

sarcasm_scores = []
for seed in [1, 2]:
    print(f"\n{'='*40}\nSeed: {seed}\n{'='*40}")

    sarcasm_model = AutoModelForSequenceClassification.from_pretrained(
        model_name, num_labels=2, problem_type="single_label_classification"
    )

    # reduce the learning rate to 2e-5
    training_args_sarcasm = TrainingArguments(
        output_dir=f'./results_sarcasm/seed_{seed}',
        num_train_epochs=5,
        seed=seed,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        learning_rate=2e-5,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='macro_f1',
        save_total_limit=1,
    )

    trainer_sarcasm = WeightedTrainer(
        model=sarcasm_model,
        args=training_args_sarcasm,
        train_dataset=train_sarcasm,
        eval_dataset=val_sarcasm,
        compute_metrics=compute_metrics,
    )

    trainer_sarcasm.train()
    predictions_sarcasm = trainer_sarcasm.predict(test_sarcasm)
    preds = np.argmax(predictions_sarcasm.predictions, axis=-1)
    score = f1_score(test_sarcasm['labels'], preds, average='macro')
    sarcasm_scores.append(score)
    print(f"Seed {seed} Macro-F1: {score:.4f}")

print(f"\nWeightedTrainer (ratio=3, lr=2e-5) Sarcasm: {np.mean(sarcasm_scores):.4f} ± {np.std(sarcasm_scores):.4f}")


In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import f1_score
import numpy as np
from datasets import load_dataset

model_name = 'microsoft/deberta-v3-base'
tokenizer = AutoTokenizer.from_pretrained(model_name)

if not 'dataset' in globals():
    dataset = load_dataset("surrey-nlp/BESSTIE-CW-26")

train_sentiment = prepare_dataset(dataset['train'], 'Sentiment')
test_sentiment = prepare_dataset(dataset['test'], 'Sentiment')
val_sentiment = prepare_dataset(dataset['validation'], 'Sentiment')

sentiment_scores = []
for seed in [1, 2, 3]:
    print(f"\n{'='*40}\nSeed: {seed}\n{'='*40}")

    # The model was loading in float16 by default. Explicitly set float32
    sentiment_model = AutoModelForSequenceClassification.from_pretrained(
        model_name,
        num_labels=2,
        problem_type="single_label_classification",
        torch_dtype=torch.float32,
    )

    training_args_seed = TrainingArguments(
        output_dir=f'./results/seed_{seed}',
        num_train_epochs=5,
        seed=seed,
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='macro_f1',
        save_total_limit=1,
    )

    trainer = Trainer(
        model=sentiment_model,
        args=training_args_seed,
        train_dataset=train_sentiment,
        eval_dataset=val_sentiment,
        compute_metrics=compute_metrics,
    )
    trainer.train()
    predictions = trainer.predict(test_sentiment)
    preds = np.argmax(predictions.predictions, axis=-1)
    score = f1_score(test_sentiment['labels'], preds, average='macro')
    sentiment_scores.append(score)
    print(f"Seed {seed} Macro-F1: {score:.4f}")

print(f"\nDeBERTa Sentiment: {np.mean(sentiment_scores):.4f} ± {np.std(sentiment_scores):.4f}")


2 seed average DeBERTa sentiment test macro-F1 is 0.9077 ± 0.0016

3 seed average DeBERTa sentiment test macro-F1 is 0.9090 ± 0.0023

In [ ]:
from sklearn.metrics import f1_score
import numpy as np

train_sarcasm = prepare_dataset(dataset['train'], 'Sarcasm')
test_sarcasm = prepare_dataset(dataset['test'], 'Sarcasm')
val_sarcasm = prepare_dataset(dataset['validation'], 'Sarcasm')

sarcasm_scores = []
for seed in [1, 2]:
    print(f"\n{'='*40}\nSeed: {seed}\n{'='*40}")

    sarcasm_model = AutoModelForSequenceClassification.from_pretrained(
        'microsoft/deberta-v3-base',
        num_labels=2,
        problem_type="single_label_classification",
        torch_dtype=torch.float32,
    )

    training_args_seed = TrainingArguments(
        output_dir=f'./results/seed_{seed}',
        num_train_epochs=5,
        seed=seed,
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='macro_f1',
        save_total_limit=1,
    )

    trainer_sarcasm = Trainer(
        model=sarcasm_model,
        args=training_args_seed,
        train_dataset=train_sarcasm,
        eval_dataset=val_sarcasm,
        compute_metrics=compute_metrics,
    )
    trainer_sarcasm.train()
    predictions_sarcasm = trainer_sarcasm.predict(test_sarcasm)
    preds = np.argmax(predictions_sarcasm.predictions, axis=-1)
    score = f1_score(test_sarcasm['labels'], preds, average='macro')
    sarcasm_scores.append(score)
    print(f"Seed {seed} Macro-F1: {score:.4f}")

print(f"\nDeBERTa Sarcasm: {np.mean(sarcasm_scores):.4f} ± {np.std(sarcasm_scores):.4f}")


In [ ]:
from sklearn.metrics import f1_score
import numpy as np

sarcasm_scores = []
for seed in [1, 2]:
    print(f"\n{'='*40}\nSeed: {seed}\n{'='*40}")

    sarcasm_model = AutoModelForSequenceClassification.from_pretrained(
        'microsoft/deberta-v3-base',
        num_labels=2,
        problem_type="single_label_classification",
        torch_dtype=torch.float32,
    )

    training_args_seed = TrainingArguments(
        output_dir=f'./results/seed_{seed}',
        num_train_epochs=10,
        seed=seed,
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='macro_f1',
        save_total_limit=1,
    )

    trainer_sarcasm = Trainer(
        model=sarcasm_model,
        args=training_args_seed,
        train_dataset=train_sarcasm,
        eval_dataset=val_sarcasm,
        compute_metrics=compute_metrics,
    )
    trainer_sarcasm.train()
    predictions_sarcasm = trainer_sarcasm.predict(test_sarcasm)
    preds = np.argmax(predictions_sarcasm.predictions, axis=-1)
    score = f1_score(test_sarcasm['labels'], preds, average='macro')
    sarcasm_scores.append(score)
    print(f"Seed {seed} Macro-F1: {score:.4f}")

print(f"\nDeBERTa Sarcasm (10 epochs): {np.mean(sarcasm_scores):.4f} ± {np.std(sarcasm_scores):.4f}")


In [ ]:
from sklearn.metrics import f1_score
import numpy as np

sarcasm_scores = []
for seed in [1, 2, 3]:
    print(f"\n{'='*40}\nSeed: {seed}\n{'='*40}")

    sarcasm_model = AutoModelForSequenceClassification.from_pretrained(
        'microsoft/deberta-v3-base',
        num_labels=2,
        problem_type="single_label_classification",
        torch_dtype=torch.float32,
    )

    training_args_seed = TrainingArguments(
        output_dir=f'./results/seed_{seed}',
        num_train_epochs=7,
        seed=seed,
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='macro_f1',
        save_total_limit=1,
    )

    trainer_sarcasm = Trainer(
        model=sarcasm_model,
        args=training_args_seed,
        train_dataset=train_sarcasm,
        eval_dataset=val_sarcasm,
        compute_metrics=compute_metrics,
    )
    trainer_sarcasm.train()
    predictions_sarcasm = trainer_sarcasm.predict(test_sarcasm)
    preds = np.argmax(predictions_sarcasm.predictions, axis=-1)
    score = f1_score(test_sarcasm['labels'], preds, average='macro')
    sarcasm_scores.append(score)
    print(f"Seed {seed} Macro-F1: {score:.4f}")

print(f"\nDeBERTa Sarcasm (7 epochs): {np.mean(sarcasm_scores):.4f} ± {np.std(sarcasm_scores):.4f}")


In [ ]:
from torch import nn
import torch
from sklearn.metrics import f1_score
import numpy as np

class DeBERTaWeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        sarcasm_ratio = len(training_dataframe[training_dataframe['Sarcasm']==0]) / len(training_dataframe[training_dataframe['Sarcasm']==1])
        weight = torch.tensor([1.0, sarcasm_ratio]).to(logits.device)
        loss_fn = nn.CrossEntropyLoss(weight=weight)
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

sarcasm_scores = []
for seed in [1, 2]:
    print(f"\n{'='*40}\nSeed: {seed}\n{'='*40}")

    sarcasm_model = AutoModelForSequenceClassification.from_pretrained(
        'microsoft/deberta-v3-base',
        num_labels=2,
        problem_type="single_label_classification",
        torch_dtype=torch.float32,
    )

    training_args_seed = TrainingArguments(
        output_dir=f'./results/seed_{seed}',
        num_train_epochs=5,
        seed=seed,
        learning_rate=1e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='macro_f1',
        save_total_limit=1,
    )

    trainer_sarcasm = DeBERTaWeightedTrainer(
        model=sarcasm_model,
        args=training_args_seed,
        train_dataset=train_sarcasm,
        eval_dataset=val_sarcasm,
        compute_metrics=compute_metrics,
    )
    trainer_sarcasm.train()
    predictions_sarcasm = trainer_sarcasm.predict(test_sarcasm)
    preds = np.argmax(predictions_sarcasm.predictions, axis=-1)
    score = f1_score(test_sarcasm['labels'], preds, average='macro')
    sarcasm_scores.append(score)
    print(f"Seed {seed} Macro-F1: {score:.4f}")

print(f"\nDeBERTa WeightedTrainer (lr=1e-5) Sarcasm: {np.mean(sarcasm_scores):.4f} ± {np.std(sarcasm_scores):.4f}")


In [ ]:
from torch import nn
import torch
from sklearn.metrics import f1_score
import numpy as np

class DeBERTaWeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        sarcasm_ratio = len(training_dataframe[training_dataframe['Sarcasm']==0]) / len(training_dataframe[training_dataframe['Sarcasm']==1])
        weight = torch.tensor([1.0, sarcasm_ratio]).to(logits.device)
        loss_fn = nn.CrossEntropyLoss(weight=weight)
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

sarcasm_scores = []
for seed in [1, 2]:
    print(f"\n{'='*40}\nSeed: {seed}\n{'='*40}")

    sarcasm_model = AutoModelForSequenceClassification.from_pretrained(
        'microsoft/deberta-v3-base',
        num_labels=2,
        problem_type="single_label_classification",
        torch_dtype=torch.float32,
    )

    training_args_seed = TrainingArguments(
        output_dir=f'./results/seed_{seed}',
        num_train_epochs=5,
        seed=seed,
        learning_rate=2e-5,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        eval_strategy='epoch',
        save_strategy='epoch',
        load_best_model_at_end=True,
        metric_for_best_model='macro_f1',
        save_total_limit=1,
    )

    trainer_sarcasm = DeBERTaWeightedTrainer(
        model=sarcasm_model,
        args=training_args_seed,
        train_dataset=train_sarcasm,
        eval_dataset=val_sarcasm,
        compute_metrics=compute_metrics,
    )
    trainer_sarcasm.train()
    predictions_sarcasm = trainer_sarcasm.predict(test_sarcasm)
    preds = np.argmax(predictions_sarcasm.predictions, axis=-1)
    score = f1_score(test_sarcasm['labels'], preds, average='macro')
    sarcasm_scores.append(score)
    print(f"Seed {seed} Macro-F1: {score:.4f}")

print(f"\nDeBERTa WeightedTrainer (lr=2e-5) Sarcasm: {np.mean(sarcasm_scores):.4f} ± {np.std(sarcasm_scores):.4f}")


Summary of DeBERTa results:

DeBERTa Sarcasm (5 epochs): 0.7175 ± 0.0025

DeBERTa Sarcasm (7 epochs): 0.7130 ± 0.0136 (3 seeds)

DeBERTa Sarcasm (10 epochs): 0.6962 ± 0.0047

DeBERTa WeightedTrainer (lr=1e-5) Sarcasm: 0.7064

DeBERTa WeightedTrainer (lr=2e-5) Sarcasm: 0.7139 ± 0.0069